In [1]:
#Cell 1: Imports
# example of defining the discriminator model
from keras.models import Sequential
from keras.optimizers import Adam
from keras.layers import Dense
from keras.layers import Conv2D
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers import LeakyReLU
from keras.utils.vis_utils import plot_model
import numpy as np
from matplotlib import pyplot

: 

In [1]:
import numpy as np
import wfdb
print("OK 1")

OK 1


In [2]:
import tensorflow as tf
print("OK 2")


[TensorFlow DLL Diagnostic] Analyzing: d:\SynGen\syn_env\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "d:\SynGen\syn_env\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [2]:
import wfdb
import os
import numpy as np

folder = r"D:/SynGen/dataset/records100/04000"

files = sorted([f for f in os.listdir(folder) if f.endswith('.hea')])

all_signals = []

for file in files:
    try:
        record_name = file.replace('.hea', '')
        path = os.path.join(folder, record_name)
        record = wfdb.rdrecord(path)
        all_signals.append(record.p_signal)
    except:
        print("Skipped:", file)

all_signals = np.array(all_signals)

# normalize
X = all_signals.astype('float32')
X = (X - X.min()) / (X.max() - X.min())
X = X * 2 - 1

dataset = X

print("Final shape:", dataset.shape)

Final shape: (1000, 1000, 12)


In [ ]:
#Cell 3: Discriminator Model
# define the standalone discriminator model
def define_discriminator(in_shape=(1000,12)):
    model = Sequential()
    
    model.add(Conv1D(64, 3, strides=2, padding='same', input_shape=in_shape))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv1D(128, 3, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv1D(256, 3, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Flatten())
    model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid'))
    
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    
    return model

In [ ]:
#Cell 4: Initialize + Plot Model
# define model
model = define_discriminator()

# summarize the model
model.summary()

# plot the model
plot_model(model, to_file='discriminator_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
#Cell 5: Load & Normalize Dataset
# function to load the dataset and scale it
def load_real_samples():
    # load cifar10 dataset
    (trainX, _), (_, _) = load_data()
    
    # convert from unsigned ints to floats
    X = trainX.astype('float32')
    
    # scale from [0,255] to [-1,1]
    X = (X - 127.5) / 127.5
    
    return X

In [ ]:
#Cell 6: Real Samples
# select real samples
def generate_real_samples(dataset, n_samples):
    # choose random instances
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    
    # retrieve selected images
    X = dataset[ix]
    
    # generate 'real' class labels (1)
    y = np.ones((n_samples, 1))
    
    return X, y

In [ ]:
#Cell 7: Fake Samples
# generate n fake samples with class labels
def generate_fake_samples(n_samples):
    # generate random noise
    X = np.random.rand(32 * 32 * 3 * n_samples)
    
    # scale to [-1,1]
    X = -1 + X * 2
    
    # reshape into images
    X = X.reshape((n_samples, 32, 32, 3))
    
    # generate 'fake' class labels (0)
    y = np.zeros((n_samples, 1))
    
    return X, y

In [ ]:
#Cell 8: Train Discriminator
# train the discriminator model temporarily
def train_discriminator(model, dataset, n_iter=20, n_batch=128):
    
    half_batch = int(n_batch / 2)
    
    for i in range(n_iter):
        
        # real samples
        X_real, y_real = generate_real_samples(dataset, half_batch)
        _, real_acc = model.train_on_batch(X_real, y_real)
        
        # fake samples
        X_fake, y_fake = generate_fake_samples(half_batch)
        _, fake_acc = model.train_on_batch(X_fake, y_fake)
        
        print('>%d real=%.0f%% fake=%.0f%%' % (i+1, real_acc*100, fake_acc*100))

In [ ]:
#Cell 9: Run Training
# define the discriminator model
model = define_discriminator()

# load image data
dataset = load_real_samples()

# fit the model
train_discriminator(model, dataset)

In [ ]:
#Cell 10: Generator Model
# define the standalone generator model
def define_generator(latent_dim):
    model = Sequential()
    
    n_nodes = 256 * 125   # 1000/8 ≈ 125
    model.add(Dense(n_nodes, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((125, 256)))
    
    model.add(Conv1DTranspose(128, 4, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv1DTranspose(128, 4, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv1DTranspose(128, 4, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    
    # output → (1000, 12)
    model.add(Conv1D(12, 3, activation='tanh', padding='same'))
    
    return model

In [ ]:
#Cell 11: Latent Space Input
# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
    # generate points in the latent space
    x_input = np.random.randn(latent_dim * n_samples)
    
    # reshape into a batch of inputs
    x_input = x_input.reshape(n_samples, latent_dim)
    
    return x_input

In [ ]:
#Cell 12: Generate Fake Samples (Using Generator)
# use the generator to generate n fake examples
def generate_fake_samples(g_model, latent_dim, n_samples):
    
    # generate points in latent space
    x_input = generate_latent_points(latent_dim, n_samples)
    
    # predict outputs
    X = g_model.predict(x_input)
    
    # create 'fake' class labels (0)
    y = np.zeros((n_samples, 1))
    
    return X, y

In [ ]:
#Cell 13: Test Generator Output
# size of the latent space
latent_dim = 100

# define generator
model = define_generator(latent_dim)

# generate samples
n_samples = 49
X, _ = generate_fake_samples(model, latent_dim, n_samples)

# scale pixel values from [-1,1] to [0,1]
X = (X + 1) / 2.0

# plot generated samples
for i in range(n_samples):
    pyplot.subplot(7, 7, 1 + i)
    pyplot.axis('off')
    pyplot.imshow(X[i])

pyplot.show()

In [ ]:
#Cell 14: Define GAN (Combine G + D)
# define the combined generator and discriminator model
def define_gan(g_model, d_model):
    
    # make discriminator not trainable
    d_model.trainable = False
    
    # connect them
    model = Sequential()
    
    # add generator
    model.add(g_model)
    
    # add discriminator
    model.add(d_model)
    
    # compile model
    opt = Adam(lr=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt)
    
    return model

In [ ]:
#Cell 15: Create GAN Model
# size of the latent space
latent_dim = 100

# create discriminator
d_model = define_discriminator()

# create generator
g_model = define_generator(latent_dim)

# create GAN
gan_model = define_gan(g_model, d_model)

# summarize model
gan_model.summary()

# plot model
plot_model(gan_model, to_file='gan_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
#Cell 16: Save Generated Images
# create and save a plot of generated images
def save_plot(examples, epoch, n=7):
    
    # scale from [-1,1] to [0,1]
    examples = (examples + 1) / 2.0
    
    # plot images
    for i in range(n * n):
        pyplot.subplot(n, n, 1 + i)
        pyplot.axis('off')
        pyplot.imshow(examples[i])
    
    # save plot
    filename = 'generated_plot_e%03d.png' % (epoch+1)
    pyplot.savefig(filename)
    pyplot.close()





In [ ]:
#Cell 17: Evaluate + Save Model
# evaluate the discriminator, plot generated images, save generator
def summarize_performance(epoch, g_model, d_model, dataset, latent_dim, n_samples=150):
    
    # real samples
    X_real, y_real = generate_real_samples(dataset, n_samples)
    _, acc_real = d_model.evaluate(X_real, y_real, verbose=0)
    
    # fake samples
    X_fake, y_fake = generate_fake_samples(g_model, latent_dim, n_samples)
    _, acc_fake = d_model.evaluate(X_fake, y_fake, verbose=0)
    
    # print accuracy
    print('>Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real*100, acc_fake*100))
    
    # save plot
    save_plot(X_fake, epoch)
    
    # save generator model
    filename = 'generator_model_%03d.h5' % (epoch+1)
    g_model.save(filename)



In [ ]:
#Cell 18: Train GAN
# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=200, n_batch=128):
    
    bat_per_epo = int(dataset.shape[0] / n_batch)
    half_batch = int(n_batch / 2)
    
    # manually enumerate epochs
    for i in range(n_epochs):
        
        # enumerate batches
        for j in range(bat_per_epo):
            
            # --------------------
            # Train Discriminator
            # --------------------
            
            # real samples
            X_real, y_real = generate_real_samples(dataset, half_batch)
            d_loss1, _ = d_model.train_on_batch(X_real, y_real)
            
            # fake samples
            X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
            d_loss2, _ = d_model.train_on_batch(X_fake, y_fake)
            
            # --------------------
            # Train Generator
            # --------------------
            
            # prepare latent points
            X_gan = generate_latent_points(latent_dim, n_batch)
            
            # inverted labels (want generator to fool discriminator)
            y_gan = np.ones((n_batch, 1))
            
            # update generator
            g_loss = gan_model.train_on_batch(X_gan, y_gan)
            
            # print progress
            print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
                  (i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))
        
        # evaluate sometimes
        if (i+1) % 10 == 0:
            summarize_performance(i, g_model, d_model, dataset, latent_dim)

In [ ]:
#Cell 19: Run Full GAN Training
# load data
dataset = load_real_samples()

# latent dimension
latent_dim = 100

# create models
d_model = define_discriminator()
g_model = define_generator(latent_dim)
gan_model = define_gan(g_model, d_model)

# train GAN
train(g_model, d_model, gan_model, dataset, latent_dim)